# Chapter 16 — Events and the Extension Lifecycle

**Companion to *Pi Agents*** · [`Pi Agents` chapter 16](https://programmer.ie/books/pi/16-chapter/)

| | |
|---|---|
| Chapter | `16-chapter.md` · weight 16 · `/books/pi/16-chapter/` |
| Notebook | `notebooks/pi/16-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

With no UI to ask, **does a `tool_call` guard leave the filesystem untouched** —
and does a handler that throws fail closed rather than open?

The chapter's point is that `tool_call` fires *before* the tool runs, so a return
value with `block: true` stops the effect. The fail-safe (a thrown handler
blocks the tool) means a buggy guard is a total outage, not an open door.


## What this notebook establishes

- A `tool_call` handler that returns `{ block: true, reason }` **prevents the tool from running** — the filesystem is untouched.
- When `ctx.hasUI` is false (print/JSON mode), the guard **refuses rather than allowing silently**.
- A `tool_call` handler that **throws blocks the tool as a fail-safe** — the same as `block: true`.
- Handlers run in registration order; `session_start` resets per-session state, not at module load.

## What this notebook does **not** establish

- **Ordering within a process only.** Two independently installed extensions racing on `tool_call` is open (`ch16-q2`).
- **A blocklist is a heuristic**, not containment. A determined model can evade it; the honest framing is that it raises the cost of an accident.
- **`agent_end` is not the end.** `agent_before_settle` is the final actionable boundary; `agent_settled` is notification-only.
- **No real `bash` is executed.** The tool call is intercepted before execution; the shell command string is the only thing examined.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real sessions with the chapter's two guards and a throwing variant, driven by a scripted model that asks for `bash`.
* **Evidence scope:** `in_process_runtime`. The shell and `git` are the reader's; the event order and block effect are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(16))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/16-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the no-UI path — the one most likely to be wrong

The chapter tests five cases. The first four run without a UI, so `ctx.hasUI` is
false and the guard must refuse rather than allow silently.

In [3]:
EVENTS = pinb.driver_source("ch16-events.ts")
events = pinb.driver(EVENTS, label="ch16-events", timeout=300)
print(pinb.md_table(
    ["case", "isError", "blocked?", "ran.flag created?"],
    [["no UI: force push", events["noUIForcePush"]["isError"], "yes" if events["noUIForcePush"]["hasGuardBlocked"] else "no", "yes" if events["noUIForcePush"]["ran"] else "no"],
     ["no UI: ordinary touch", events["noUIOrdinary"]["isError"], "no", "yes" if events["noUIOrdinary"]["ran"] else "no"],
     ["no UI: reset --hard", events["noUIResetHard"]["isError"], "yes", "-"],
     ["no UI: multi-hook guard", events["noUIMultiHook"]["isError"], "yes" if events["noUIMultiHook"]["hasForceWithLease"] else "no", "yes" if events["noUIMultiHook"]["ran"] else "no"]
    ]))


| case | isError | blocked? | ran.flag created? |
|---|---|---|---|
| no UI: force push | True | yes | no |
| no UI: ordinary touch | False | no | yes |
| no UI: reset --hard | True | yes | - |
| no UI: multi-hook guard | True | yes | no |


In [4]:
pinb.show_checks([
    pinb.check("no UI: force push is blocked and ran.flag is NOT created",
               events["noUIForcePush"]["isError"] and events["noUIForcePush"]["hasGuardBlocked"] and not events["noUIForcePush"]["ran"],
               "blocked, no side effect"),
    pinb.check("no UI: ordinary command runs and ran.flag IS created",
               not events["noUIOrdinary"]["isError"] and events["noUIOrdinary"]["ran"],
               "allowed, side effect present"),
    pinb.check("no UI: reset --hard is blocked",
               events["noUIResetHard"]["isError"], "blocked"),
    pinb.check("no UI: multi-hook guard blocks with --force-with-lease reason",
               events["noUIMultiHook"]["isError"] and events["noUIMultiHook"]["hasForceWithLease"] and not events["noUIMultiHook"]["ran"],
               "blocked, no side effect"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------------  ----------------------------
PASS  no UI: force push is blocked and ran.flag is NOT created  blocked, no side effect
PASS  no UI: ordinary command runs and ran.flag IS created  allowed, side effect present
PASS  no UI: reset --hard is blocked  blocked
PASS  no UI: multi-hook guard blocks with --force-with-lease reason  blocked, no side effect

4/4 assertions held.


## With a UI: the person decides

When `ctx.hasUI` is true, the guard asks. Two runs: approve and decline.

In [5]:
print(pinb.md_table(
    ["case", "isError", "ran.flag created?"],
    [["with UI: approved", events["withUIApproved"]["isError"], "yes" if events["withUIApproved"]["ran"] else "no"],
     ["with UI: declined", events["withUIDeclined"]["isError"], "yes" if events["withUIDeclined"]["ran"] else "no"]],
))

| case | isError | ran.flag created? |
|---|---|---|
| with UI: approved | False | yes |
| with UI: declined | True | no |


In [6]:
pinb.show_checks([
    pinb.check("with UI: approved runs the command",
               not events["withUIApproved"]["isError"] and events["withUIApproved"]["ran"],
               "allowed, side effect present"),
    pinb.check("with UI: declined blocks and ran.flag is NOT created",
               events["withUIDeclined"]["isError"] and not events["withUIDeclined"]["ran"],
               "blocked, no side effect"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------  ----------------------------
PASS  with UI: approved runs the command  allowed, side effect present
PASS  with UI: declined blocks and ran.flag is NOT created  blocked, no side effect

2/2 assertions held.


## Fail-safe: a throwing handler blocks the tool

`extensions.md` says: a `tool_call` handler failure blocks the tool as a
fail-safe. This is observed, not just documented.

In [7]:
print(pinb.md_table(
    ["case", "isError", "ran.flag created?"],
    [["throwing handler", events["throwing"]["isError"], "yes" if events["throwing"]["ran"] else "no"]],
))

| case | isError | ran.flag created? |
|---|---|---|
| throwing handler | True | no |


In [8]:
pinb.show_checks([
    pinb.check("a throwing tool_call handler blocks the tool (fail-safe)",
               events["throwing"]["isError"] and not events["throwing"]["ran"],
               "blocked, no side effect"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------  -----------------------
PASS  a throwing tool_call handler blocks the tool (fail-safe)  blocked, no side effect

1/1 assertions held.


## The chapter's own tests

The five cases above are the chapter's test file, executed unmodified.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [9]:
PATTERNS = 'ch16-events/guard.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch16-events/guard.test.ts', show="blocked")

$ node --test ch16-events/guard.test.ts
  PASS  5 passed, 0 failed, 0 skipped  in 989 ms

  tests matching 'blocked':


## Your turn: predict, then run

**Predict first.** Add a second guard extension that also blocks `bash` on a
different pattern. Which one sees the call first — and what happens if the first
one allows but the second blocks?

**Then change one input.** The regexes match against `JSON.stringify(event.input)`.
What happens if the model sends `{"command": "git push --force"}` vs
`{"command": "git push --force-with-lease"}` — does the second match the
`--force|-f` pattern?

**Predict the boundary.** The throwing handler is a fail-safe. What happens if
*every* `tool_call` handler in the stack throws? Is the first one the one that
wins, or does the error propagate?

In [10]:
print("Predictions:")
print("  1. Handlers run in registration order; the first to return block wins.")
print("  2. --force-with-lease does NOT match the --force|-f pattern.")
print("  3. The first handler to throw wins — the error stops the chain.")
print()
print("Observed above:")
print(f"  no UI force push: isError={events["noUIForcePush"]["isError"]}, blocked={events["noUIForcePush"]["hasGuardBlocked"]}, ran={events["noUIForcePush"]["ran"]}")
print(f"  throwing handler:   isError={events["throwing"]["isError"]}, ran={events["throwing"]["ran"]}")
print()
assert events["noUIForcePush"]["isError"] and not events["noUIForcePush"]["ran"]
assert events["throwing"]["isError"] and not events["throwing"]["ran"]
print("held: the guard refuses without UI, and a thrown handler fails closed")

Predictions:
  1. Handlers run in registration order; the first to return block wins.
  2. --force-with-lease does NOT match the --force|-f pattern.
  3. The first handler to throw wins — the error stops the chain.

Observed above:
  no UI force push: isError=True, blocked=True, ran=False
  throwing handler:   isError=True, ran=False

held: the guard refuses without UI, and a thrown handler fails closed


## Interpretation

An event handler is three things, and the chapter shows all three:

1. **A gate** — `tool_call` with `block: true` stops the effect before it runs.
2. **A dialog** — `ctx.ui.confirm` asks when `ctx.hasUI`; print/JSON mode has no
   UI, so the honest behaviour is to refuse.
3. **A fail-safe** — an unhandled error in `tool_call` blocks the tool, so a
   buggy guard is a total outage, not an open door.

The practical rules:

* **Use the event's declared result type.** `tool_call` returns `{ block?, reason?, terminate? }` — nothing else has an effect.
* **Reset per-session state in `session_start`, not at module load.** A module-level variable survives reload; `session_start` fires once per session.
* **Catch inside the handler.** A guard that throws on every call is a total outage.
* **Handlers run in registration order.** If another extension also guards `bash`, whichever loaded first sees the call first.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real sessions with the chapter's two guards and a throwing variant, driven by a scripted model that asks for `bash`.

**Evidence scope:** `in_process_runtime`. The shell and `git` are the reader's; the event order and block effect are Pi's.

### What was read or run

- **Ran** `drivers/ch16-events.ts`: seven real sessions with `ch16-events/guard.ts`, `ch16-events/guard-status.ts`, and a throwing variant.
- **Ran** `ch16-events/guard.test.ts` (5 tests).
- **Read** `examples/evidence.json`, chapter 16 rows (5 claims).

### Limitations

- **Ordering within a process only.** Two independently installed extensions racing on `tool_call` is not exercised.
- **A blocklist is a heuristic, not containment.** The regexes match the serialised input; a determined model can evade them.
- **No real `bash` is executed.** The tool call is intercepted before execution; only the command string is examined.
- **`agent_end` is not the end** — `agent_before_settle` is the final actionable boundary. This is documented, not run here.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
